# EDA - Medicamentos Vitales No Disponibles

Este notebook realiza un análisis exploratorio inicial sobre el dataset limpio generado por el pipeline de validación.

Objetivos del notebook:
- revisar estructura del dataset limpio
- explorar comportamiento temporal
- identificar principios activos más frecuentes
- analizar tipos de solicitud
- revisar principales importadores
- observar duplicados sospechosos


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR = BASE_DIR / 'data' / 'processed'

clean_path = DATA_DIR / 'dataset_clean.csv'
duplicates_path = DATA_DIR / 'suspected_duplicates.csv'
quality_path = DATA_DIR / 'quality_summary.csv'

df = pd.read_csv(clean_path)
df_dup = pd.read_csv(duplicates_path) if duplicates_path.exists() else pd.DataFrame()
quality = pd.read_csv(quality_path) if quality_path.exists() else pd.DataFrame()

print('Dataset limpio:', df.shape)
print('Duplicados sospechosos:', df_dup.shape)
quality


## 1. Vista general del dataset

In [ ]:
df.head()

In [ ]:
df.info()

## 2. Preparación de columnas auxiliares

In [ ]:
df['fecha_de_autorizacion'] = pd.to_datetime(df['fecha_de_autorizacion'], errors='coerce')
df['anio'] = df['fecha_de_autorizacion'].dt.year
df['mes'] = df['fecha_de_autorizacion'].dt.month

if 'cantidad_solicitada' in df.columns:
    df['cantidad_solicitada'] = pd.to_numeric(df['cantidad_solicitada'], errors='coerce')

df[['fecha_de_autorizacion', 'anio', 'mes']].head()


## 3. Solicitudes por año

In [ ]:
solicitudes_anio = df['anio'].value_counts().sort_index()
solicitudes_anio

In [ ]:
plt.figure(figsize=(10, 5))
solicitudes_anio.plot(kind='bar')
plt.title('Solicitudes por año')
plt.xlabel('Año')
plt.ylabel('Cantidad de solicitudes')
plt.tight_layout()
plt.show()

## 4. Tipos de solicitud

In [ ]:
tipo_solicitud = df['tipo_de_solicitud'].value_counts(dropna=False)
tipo_solicitud

In [ ]:
plt.figure(figsize=(8, 5))
tipo_solicitud.plot(kind='bar')
plt.title('Distribución por tipo de solicitud')
plt.xlabel('Tipo de solicitud')
plt.ylabel('Cantidad')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## 5. Principios activos más frecuentes

In [ ]:
top_principios = df['principio_activo1'].value_counts().head(15)
top_principios

In [ ]:
plt.figure(figsize=(12, 6))
top_principios.sort_values().plot(kind='barh')
plt.title('Top 15 principios activos más frecuentes')
plt.xlabel('Número de registros')
plt.ylabel('Principio activo')
plt.tight_layout()
plt.show()

## 6. Importadores más frecuentes

In [ ]:
top_importadores = df['solicitante_importador'].value_counts().head(15)
top_importadores

In [ ]:
plt.figure(figsize=(12, 6))
top_importadores.sort_values().plot(kind='barh')
plt.title('Top 15 solicitantes/importadores')
plt.xlabel('Número de registros')
plt.ylabel('Importador')
plt.tight_layout()
plt.show()

## 7. Cantidad solicitada

In [ ]:
df['cantidad_solicitada'].describe()

In [ ]:
plt.figure(figsize=(8, 5))
df['cantidad_solicitada'].dropna().plot(kind='hist', bins=30)
plt.title('Distribución de cantidad solicitada')
plt.xlabel('Cantidad solicitada')
plt.ylabel('Frecuencia')
plt.tight_layout()
plt.show()

## 8. Duplicados sospechosos

In [ ]:
print('Cantidad de filas marcadas como duplicadas:', len(df_dup))
df_dup.head()

## 9. Conclusiones preliminares

- El pipeline logró una base limpia con un porcentaje de rechazo bajo.
- La mayoría de registros corresponden a unos pocos principios activos prioritarios.
- Existen importadores con alta concentración de solicitudes.
- El análisis temporal puede apoyar decisiones de seguimiento y abastecimiento.
- Los duplicados sospechosos deben interpretarse como casos para revisión, no como errores automáticamente eliminables.
